# Week 5: Fourier Pricing (Carr-Madan, COS)

## Why we're here

Everything built in Week 4 (QE variance sampler, correlated path simulator) is
Monte Carlo. It works, but convergence is slow ($O(1/\sqrt N)$) and every parameter
tweak means re-simulating from scratch, expensive if this is ever going to feed a
calibration loop (Week 6). The Heston characteristic function derived at the end of
Week 4,

$$\phi(u;\tau) = \exp\big(iu\ln S_0 + C(\tau;u) + D(\tau;u)\,v_0\big)$$

gives a semi-closed-form route instead: Fourier inversion techniques price an
entire strip of strikes almost instantly, no simulation required. This week covers
two such techniques, Carr-Madan (the original, FFT-based) and COS (the more modern
alternative), both consuming `heston_char_func` as a black-box input.

## The COS method

### Step 1: truncate the domain

The risk-neutral density of $x_T = \ln S_T$ lives, in principle, on all of
$\mathbb R$, but in practice it is negligibly small outside some finite range
$[a,b]$ (chosen from the model's cumulants, mean/variance/skewness derived from
Heston's parameters). Restricting to a finite interval is what makes a *cosine*
series, as opposed to a full Fourier series over all of $\mathbb R$, applicable at
all, cosine series are naturally suited to finite intervals.

### Step 2: the cosine-series identity

Any reasonably well-behaved function $f(x)$ on $[a,b]$ can be written as

$$f(x) = \sum_{k=0}^{\infty}{}' A_k \cos\left(k\pi\frac{x-a}{b-a}\right)$$

(the prime denotes that the $k=0$ term carries weight $\tfrac12$), where the
coefficients $A_k$ are recoverable from $f$ via an integral. This much is standard
Fourier analysis, nothing Heston-specific yet.

### Step 3: coefficients from the characteristic function directly

Since $f$ is a *probability density* here (the density of $x_T$), its cosine
coefficients $A_k$ turn out to be expressible directly in terms of the
characteristic function evaluated at specific points $u_k = \frac{k\pi}{b-a}$:

$$A_k \approx \frac{2}{b-a}\,\text{Re}\left[\phi\!\left(\frac{k\pi}{b-a}\right)\exp\left(-i\frac{k\pi a}{b-a}\right)\right]$$

This is the entire payoff of the method: no numerical integration is needed to get
these coefficients, just plug $u_k$ into `heston_char_func` and take the real part.
This is why COS tends to be faster and more numerically robust than Carr-Madan in
practice, Carr-Madan needs an FFT over a damped, oscillatory integrand and a choice
of damping factor; COS evaluates $\phi$ at a handful of points and sums.

### Step 4: pricing

The option price is an expectation of a payoff against this density, $E[g(x_T)]$.
With $f(x)\approx\sum A_k\cos(\ldots)$, and the payoff $g$ (e.g. $(e^x-K)^+$ for a
call) having its own cosine-series coefficients $V_k$ over the same interval
(closed forms exist for vanilla payoffs), the price collapses to a finite sum:

$$\text{price} \approx e^{-r\tau}\sum_{k=0}^{N-1}{}' A_k V_k$$

No integration, no FFT, only $N$ characteristic-function evaluations and a dot
product.

## Plan for this week

1. `heston_char_func` in `models/heston.py` - **done** (Day 21), branch-safe
   Riccati root (Albrecher et al. 2007, "The Little Heston Trap") to avoid the
   discontinuous complex-log failure mode of the naive Heston (1993) formulation
2. COS method implementation: domain truncation $[a,b]$ from cumulants, payoff
   coefficients $V_k$ for a call, assembly into a pricer
3. Carr-Madan method: damping factor, FFT-based inversion
4. Validation: price a strip of strikes via both Fourier methods, cross-check
   against Week 4's Monte Carlo simulator as ground truth, convergence-rate checks
   (more cosine terms / finer FFT grid -> error shrinking at the expected rate)
   rather than single-point comparisons, same discipline as Dupire's checkpoints

## Note carried over from Week 4

The characteristic function's Riccati equation is structurally the same object as
the Riccati equations in LQR control and Kalman filter covariance propagation,
quadratic-in-the-unknown, backward-in-time, arising from a linear-quadratic
cost/variance object propagated via Feynman-Kac-adjacent machinery. Same reduction
principle, different application.
